# Showcasing Row Level Security (RLS)
Unity Catalog allows for the definition of Row-Level-Security rules, which can use generic SQL Logic to determine whether a given user can/cannot see specific rows.

In [0]:
schema_teste = "cedip.default"

In [0]:
# Create a DataFrame with sample data
data = [
    ("A", "2025-07-08", 100),
    ("A", "2025-07-09", 150),
    ("A", "2025-07-10", 200),
    ("B", "2025-07-08", 110),
    ("B", "2025-07-09", 160),
    ("B", "2025-07-10", 210),
    ("C", "2025-07-08", 120),
    ("C", "2025-07-09", 170),
    ("C", "2025-07-10", 220)
]

columns = ["group_access_control", "date", "amount"]

table_name = f"{schema_teste}.sales"
func_name = f"{schema_teste}.sales_func"
df = spark.createDataFrame(data, columns)

# Create the table in 'schema_teste' and insert the data
spark.sql(f"DROP TABLE IF EXISTS {table_name}")
df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_name)

# Display the table to verify
display(spark.table(table_name))

In [0]:
%sql
SELECT
    is_account_group_member("TEST_GROUP_A"),
    is_account_group_member("TEST_GROUP_B"),
    is_account_group_member("TEST_GROUP_C")

In [0]:
spark.sql(f"""
SELECT 
    * 
FROM {table_name}
WHERE
    (is_account_group_member("TEST_GROUP_A") AND group_access_control = "A")
    OR (is_account_group_member("TEST_GROUP_B") AND group_access_control = "B")
    OR (is_account_group_member("TEST_GROUP_C") AND group_access_control = "C")
""").display()

In [0]:
spark.sql(f"""
CREATE OR REPLACE FUNCTION {func_name}(group_access_control STRING)
RETURN
    (is_account_group_member("TEST_GROUP_A") AND group_access_control = "A")
    OR (is_account_group_member("TEST_GROUP_B") AND group_access_control = "B")
    OR (is_account_group_member("TEST_GROUP_C") AND group_access_control = "C")
;
""")

In [0]:
spark.sql(f"ALTER TABLE {table_name} DROP ROW FILTER;")
spark.sql(f"ALTER TABLE {table_name} SET ROW FILTER {func_name} ON (group_access_control);")

In [0]:
spark.sql(f"""
SELECT 
    * 
FROM {table_name}
""").display()